In [10]:
!pip install prophet

In [1]:
!pip install keras_tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.4 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import keras_tuner as kt

#os.makedirs('../results', exist_ok=True)

df_15min = pd.read_csv('/content/datos_procesados.csv', index_col='fecha', parse_dates=True)
print(f"Filas cargadas: {len(df_15min)}")

Filas cargadas: 441


Funciones N-BEATS

In [4]:
def nbeats_block(x, units, backcast_length, forecast_length):
    for _ in range(4):
        x = layers.Dense(units, activation='relu')(x)
    backcast = layers.Dense(backcast_length)(x)
    forecast = layers.Dense(forecast_length)(x)
    return backcast, forecast

def build_nbeats(input_length=24, forecast_length=4, units=256, n_blocks=3):
    inputs = keras.Input(shape=(input_length,))
    residual = inputs
    total_forecast = None
    for _ in range(n_blocks):
        backcast, forecast = nbeats_block(residual, units, input_length, forecast_length)
        residual = layers.Subtract()([residual, backcast])
        total_forecast = forecast if total_forecast is None else layers.Add()([total_forecast, forecast])
    return keras.Model(inputs=inputs, outputs=total_forecast)

def create_sequences(data, input_length=24, forecast_length=4):
    X, y = [], []
    for i in range(len(data) - input_length - forecast_length + 1):
        X.append(data[i:i+input_length])
        y.append(data[i+input_length:i+input_length+forecast_length])
    return np.array(X), np.array(y)

Scaler y Tuner

In [5]:
scaler = MinMaxScaler()
values = df_15min['body.counting'].values.reshape(-1, 1)
values_scaled = scaler.fit_transform(values).flatten()

callbacks_tuner = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

best_overall_loss = np.inf
best_overall_model = None
best_overall_hp = {}

for input_length in [24, 48, 96]:
    X_t, y_t = create_sequences(values_scaled, input_length=input_length)
    split = int(len(X_t) * 0.8)
    X_train_t, X_val_t = X_t[:split], X_t[split:]
    y_train_t, y_val_t = y_t[:split], y_t[split:]

    def build_model_tunable(hp):
        units         = hp.Choice('units', values=[32, 64])
        n_blocks      = hp.Int('n_blocks', min_value=1, max_value=3)
        learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
        model = build_nbeats(input_length=input_length, forecast_length=4, units=units, n_blocks=n_blocks)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
        return model

    tuner = kt.BayesianOptimization(
        build_model_tunable,
        objective='val_loss',
        max_trials=20,
        num_initial_points=5,
        directory='../results/nbeats_tuning',
        project_name=f'trafico_il{input_length}',
        overwrite=True
    )

    tuner.search(
        X_train_t, y_train_t,
        validation_data=(X_val_t, y_val_t),
        epochs=100,
        callbacks=callbacks_tuner,
        verbose=0
    )

    trial_loss = tuner.oracle.get_best_trials(1)[0].score
    print(f"input_length={input_length:3d} → mejor val_loss: {trial_loss:.6f}")

    if trial_loss < best_overall_loss:
        best_overall_loss  = trial_loss
        best_overall_hp    = tuner.get_best_hyperparameters(1)[0]
        best_overall_model = tuner.get_best_models(1)[0]
        best_input_length  = input_length
        X_best_train, y_best_train = X_train_t, y_train_t
        X_best_val,   y_best_val   = X_val_t,   y_val_t

print(f"""
Mejores hiperparámetros:
  input_length  : {best_input_length}
  units         : {best_overall_hp.get('units')}
  n_blocks      : {best_overall_hp.get('n_blocks')}
  learning_rate : {best_overall_hp.get('learning_rate')}
""")

input_length= 24 → mejor val_loss: 0.011580


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 46 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


input_length= 48 → mejor val_loss: 0.012646


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


input_length= 96 → mejor val_loss: 0.012864

Mejores hiperparámetros:
  input_length  : 24
  units         : 64
  n_blocks      : 2
  learning_rate : 0.01



Entrenamiento

In [7]:
best_overall_model.fit(
    X_best_train, y_best_train,
    validation_data=(X_best_val, y_best_val),
    epochs=200,
    batch_size=32,
    callbacks=callbacks_tuner,
    verbose=1
)

Epoch 1/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0124 - val_loss: 0.0122 - learning_rate: 0.0100
Epoch 2/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0113 - val_loss: 0.0123 - learning_rate: 0.0100
Epoch 3/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0108 - val_loss: 0.0121 - learning_rate: 0.0100
Epoch 4/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0104 - val_loss: 0.0126 - learning_rate: 0.0100
Epoch 5/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0094 - val_loss: 0.0118 - learning_rate: 0.0100
Epoch 6/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0092 - val_loss: 0.0124 - learning_rate: 0.0100
Epoch 7/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0089 - val_loss: 0.0137 - learning_rate: 0.0100
Epoch 8/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0085 - val_loss: 0.0123 - learning_rate: 0.0100
Epoch 9/200
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0089 - val_loss: 0.0128 - learning_rate: 0.0100
Epoch 10/200
11

Guardado de modelos

In [8]:
best_overall_model.save('../results/nbeats_trafico.keras')

metadata = {
    'best_input_length': best_input_length,
    'forecast_length': 4,
    'units': best_overall_hp.get('units'),
    'n_blocks': best_overall_hp.get('n_blocks'),
    'learning_rate': best_overall_hp.get('learning_rate'),
}

with open('../results/nbeats_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../results/nbeats_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print("Guardado en ../results/")

Guardado en ../results/


Segundo modelo: Prophet


In [11]:
from prophet import Prophet
import pandas as pd
import numpy as np

# ── Preparar datos en formato Prophet (ds, y) ─────────────────────
df_prophet = df_15min.reset_index().rename(columns={'fecha': 'ds', 'body.counting': 'y'})

# ── Split cronológico 80/20 ───────────────────────────────────────
split = int(len(df_prophet) * 0.8)
train_prophet = df_prophet[:split]
test_prophet  = df_prophet[split:]

# ── Modelo ───────────────────────────────────────────────────────
model_prophet = Prophet(
    seasonality_mode='additive',
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=False  # solo tenemos 1 semana de datos
)
model_prophet.fit(train_prophet)

# ── Predicción sobre el período de test ──────────────────────────
future = model_prophet.make_future_dataframe(periods=len(test_prophet), freq='15min')
forecast = model_prophet.predict(future)

# ── Extraer solo el período de test ──────────────────────────────
forecast_test = forecast.tail(len(test_prophet)).reset_index(drop=True)
y_real_prophet = test_prophet['y'].values
y_pred_prophet = forecast_test['yhat'].values.clip(min=0)  # no hay vehículos negativos

# ── Métricas ─────────────────────────────────────────────────────
mae_p  = np.mean(np.abs(y_real_prophet - y_pred_prophet))
rmse_p = np.sqrt(np.mean((y_real_prophet - y_pred_prophet) ** 2))
mape_p = np.mean(np.abs((y_real_prophet - y_pred_prophet) / (y_real_prophet + 1e-8))) * 100

print(f"Prophet — MAE: {mae_p:.2f} | RMSE: {rmse_p:.2f} | MAPE: {mape_p:.2f}%")

# ── Guardar modelo ────────────────────────────────────────────────
import pickle
with open('../results/prophet_model.pkl', 'wb') as f:
    pickle.dump(model_prophet, f)

Prophet — MAE: 9.52 | RMSE: 12.59 | MAPE: 100.84%


In [12]:
smape_p = np.mean(2 * np.abs(y_real_prophet - y_pred_prophet) /
                  (np.abs(y_real_prophet) + np.abs(y_pred_prophet) + 1e-8)) * 100
print(f"Prophet — sMAPE: {smape_p:.2f}%")

Prophet — sMAPE: 54.75%


In [13]:
from google.colab import files
import shutil

shutil.make_archive('results', 'zip', '../results')
files.download('results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>